![](https://github.com/DA4BAM/image/blob/main/step%20by%20step.png?raw=true)

![](https://github.com/DA4BAM/image/blob/main/pipeline%20function.png?raw=true)

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, Dropout

In [3]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

In [4]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

## 전처리 과정

In [5]:
train_data.drop('subject', axis=1, inplace=True)

In [6]:
X_train = train_data.drop('Activity', axis=1)

### feature_importance_를 사용한 모델이 있는 경우 실행하는 구간

In [7]:
# feature_importance를 사용한 모델이 있는 경우 이 코드를 실행.
rf_model_for_fi = joblib.load('rf_model_gs(중요 피처를 뽑기 위한 모델).pkl')

In [8]:
RFmodel_fi_df = plot_feature_importance(rf_model_for_fi.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,tBodyAccMag-std(),0.036021
1,tGravityAccMag-std(),0.032272
2,"tGravityAcc-arCoeff()-Z,2",0.028924
3,fBodyAccMag-std(),0.028493
4,fBodyAccMag-energy(),0.027746
...,...,...
556,"fBodyAcc-bandsEnergy()-49,64.1",0.000022
557,fBodyAcc-min()-Z,0.000020
558,"fBodyAcc-bandsEnergy()-33,48.1",0.000019
559,"fBodyAccJerk-bandsEnergy()-49,64",0.000016


In [9]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [10]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [11]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features_fi = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features_fi)}")

누적 중요도 95%까지 선택된 변수 개수: 287


In [12]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features_fi), 'feature_importance'].sum()
print(f"상위 {len(selected_features_fi)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 288개 변수의 누적 중요도: 0.9502


In [13]:
RFmodel_fi_df['feature_name'][:len(selected_features_fi)+1]

,feature_name
0,tBodyAccMag-std()
1,tGravityAccMag-std()
2,"tGravityAcc-arCoeff()-Z,2"
3,fBodyAccMag-std()
4,fBodyAccMag-energy()
...,...
283,fBodyAccMag-kurtosis()
284,tGravityAcc-iqr()-Z
285,"fBodyAccJerk-bandsEnergy()-33,48.2"
286,tBodyGyroJerkMag-std()


### permutation importance를 사용한 모델이 있는 경우 실행하는 구간

In [14]:
# permutation importance 데이터프레임 불러오기
importances_df = pd.read_csv('svm_permutation_importance_DataFrame(그리드서치).csv')

In [15]:
selected_features_pi = importances_df['feature']

In [16]:
selected_features_pi

,feature
0,tBodyGyroJerk-entropy()-X
1,tGravityAcc-mean()-Y
2,tGravityAcc-max()-Y
3,tGravityAcc-min()-Y
4,fBodyGyro-entropy()-X
...,...
193,tBodyGyroMag-max()
194,tBodyGyroMag-mad()
195,fBodyBodyGyroMag-iqr()
196,"tBodyGyroJerk-arCoeff()-Z,4"


### 레이블 인코딩이 필요한 경우 실행하는 구간

In [ ]:
# is_dynamic = {
#     'STANDING': 0,
#     'SITTING': 0,
#     'LAYING': 0,
#     'WALKING': 1,
#     'WALKING_UPSTAIRS': 1,
#     'WALKING_DOWNSTAIRS': 1
# }

In [ ]:
# test_data['Activity_dynamic'] = test_data['Activity'].map(is_dynamic)

In [ ]:
# y_test = test_data.loc[:, 'Activity_dynamic']

In [ ]:
# isDynamic_le = LabelEncoder()

In [ ]:
# dynamic_le = LabelEncoder()
# dynamic_le.fit_transform()

In [ ]:
# static_le = LabelEncoder()
# static_le.fit_transform()

### 각각의 베스트 모델을 불러옴

In [43]:
#isDynamic_model = joblib.load('svm_model_gs.pkl')
isDynamic_model = load_model('cnn_model(2D)_static_vs_dynamic.h5')
dynamic_model = joblib.load('svm_model_gs_fi.pkl')
static_model = joblib.load('svm_model_gs_pi.pkl')

In [44]:
def activity_classifier_model_usingSaveModel(test_data):
    final_predict = []

    test_data.dropna(inplace=True)
    test_data.reset_index(drop=True, inplace=True)

    X_test = test_data.drop(['subject', 'Activity'], axis=1)


    y_test = test_data.loc[:, 'Activity']
    X_test_reshaped = X_test.values.reshape(-1, 33, 17, 1) # CNN 2D모델일 때 실행
    isDynamic_pred = isDynamic_model.predict(X_test_reshaped)  # CNN 2D모델일 때: X_test_reshaped를 넣고 실행
    isDynamic_pred = (isDynamic_pred > 0.5).astype(int) # CNN 모델일 때 실행

    for idx in range(len(isDynamic_pred)):
        if(isDynamic_pred[idx] == 1): # 동적vs정적 분류 모델이 만약 0,1의 레이블이라면 'DYNAMIC'을 1로 교체
            pred = dynamic_model.predict(X_test.loc[[idx], RFmodel_fi_df['feature_name'][:len(selected_features_fi)+1]])
            # if isinstance(pred[0], (int, np.integer)):
            #     pred = dynamic_le.inverse_transform(pred)
        else:
            pred = static_model.predict(X_test.loc[[idx], selected_features_pi])
            # if isinstance(pred[0], (int, np.integer)):
            #     pred = static_le.inverse_transform(pred)

        final_predict.append(pred[0])

    print('accuracy :',accuracy_score(y_test, final_predict))
    print('='*60)
    print(confusion_matrix(y_test, final_predict))
    print('='*60)
    print(classification_report(y_test, final_predict))
    print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, final_predict):.4f}")

    return pd.Series(final_predict, name='predict_result')

In [45]:
predict_result = activity_classifier_model_usingSaveModel(test_data)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
accuracy : 0.991162474507138
[[292   0   0   0   0   0]
 [  0 248   6   0   0   0]
 [  0   6 281   0   0   0]
 [  0   0   0 228   0   0]
 [  0   0   0   0 195   0]
 [  0   0   0   1   0 214]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       292
           SITTING       0.98      0.98      0.98       254
          STANDING       0.98      0.98      0.98       287
           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           0.99      1471
         macro avg       0.99      0.99      0.99      1471
      weighted avg       0.99      0.99      0.99      1471

Cohen's Kappa Score: 0.9893


In [46]:
predict_result

,predict_result
0,SITTING
1,STANDING
2,WALKING
3,SITTING
4,STANDING
...,...
1466,SITTING
1467,STANDING
1468,WALKING_DOWNSTAIRS
1469,LAYING
